## 1. Install deps & mount Drive

In [3]:
!pip install -q openai beautifulsoup4 requests chromadb python-dotenv pandas "google-search-results" tiktoken sentence-transformers

import os
from pathlib import Path

# Colab: mount Drive and use persistent project root
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/research")
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    PROJECT_ROOT = Path(".").resolve()

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / "data").mkdir(exist_ok=True)
(PROJECT_ROOT / "cis").mkdir(exist_ok=True)
(PROJECT_ROOT / "cis" / "raw").mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_ROOT)
print("Project root:", PROJECT_ROOT, "| Colab:", IN_COLAB)

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 74.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━

## 2. API keys & LLM backend (huggingface now; switch to "openai" when you have the key)

In [4]:
try:
    from google.colab import userdata

    try:
        raw_key = userdata.get("OPENAI_API_KEY")
        if raw_key:
            # FIX: .strip() removes hidden newlines that cause "Illegal header" errors
            os.environ["OPENAI_API_KEY"] = raw_key.strip()
    except Exception:
        pass

    # Load SERPAPI_API_KEY (Optional)
    try:
        raw_serp = userdata.get("SERPAPI_API_KEY")
        if raw_serp:
            os.environ["SERPAPI_API_KEY"] = raw_serp.strip()
    except Exception:
        pass

except Exception as e:
    print(f"⚠️ Error accessing Colab Secrets: {e}")

# 2. Validate Critical Keys
if not os.environ.get("OPENAI_API_KEY"):
    print("❌ ERROR: OPENAI_API_KEY is missing! Please add it to the Secrets tab (Key icon on the left).")
else:
    print(f"✅ OPENAI_API_KEY loaded: ***{os.environ['OPENAI_API_KEY'][-4:]}")

print(f"✓ SERPAPI_API_KEY: {'Set (Google Bot Enabled)' if os.environ.get('SERPAPI_API_KEY') else 'Not Set (Google Bot Disabled)'}")

✅ OPENAI_API_KEY loaded: ***GKsA
✓ SERPAPI_API_KEY: Set (Google Bot Enabled)


## 3. Config

In [5]:
DATA_DIR = PROJECT_ROOT / "data"
CIS_RAW = PROJECT_ROOT / "cis" / "raw"
CIS_INDEX = PROJECT_ROOT / "cis" / "chroma_index"
QUESTIONS_PATH = DATA_DIR / "questions_62_ja.json"
RESULTS_PATH = DATA_DIR / "results_372.json"

GPT4, GPT35 = "gpt-4", "gpt-3.5-turbo-16k"
EMBED_MODEL = "text-embedding-ada-002"
RAG_INSTRUCTION = (
    "Use the following pieces of context to answer the question. "
    "If you don't know the answer, just say that you don't know; "
    "don't try to make up an answer."
)
TOP_K_CIS, TOP_K_GOOGLE = 5, 10

# --- Reranking Settings ---
USE_RERANKING = True
TOP_N_INITIAL = 20  # Retrieve this many first
TOP_K_FINAL = 5     # Rerank and keep this many for the LLM
RERANK_MODEL_NAME = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"

CIS_START = "https://ganjoho.jp/public/index.html"

# CIS: lite = cap pages (Colab-friendly). Full: set LITE_CRAWL=False.
LITE_CRAWL = True
MAX_PAGES_LITE = 80

# Set False to run only Conventional + Google (no CIS, no scraping).
RUN_CIS = True


## 4. Questions (upload `data/questions_62_ja.json` to Drive, or use minimal below)

In [ ]:
# --- STEP 3: Load Questions (Robust Fix) ---
import json

# Minimal fallback data (used only if file is missing)
MINIMAL_QUESTIONS = [
    {"id": 1, "text_ja": "私は札幌市北区に住んでいます。友人が胆管がんと診断されました。どの病院で治療を受けるのがおすすめでしょうか。", "cis_covered": True},
    {"id": 2, "text_ja": "赤酵母米のサプリメントはがんを悪化させる可能性がありますか。", "cis_covered": True},
    {"id": 3, "text_ja": "千葉市に重粒子線治療を行う病院があると聞き、主治医に相談したが紹介してくれなかった。なぜか。", "cis_covered": True},
]

if QUESTIONS_PATH.exists():
    try:
        # 1. Load the raw file
        text = QUESTIONS_PATH.read_text(encoding="utf-8")
        raw_data = json.loads(text)

        # 2. Handle nested structure (Fixes the "string indices must be integers" error)
        if isinstance(raw_data, dict):
            # Try to find the list inside the dictionary
            if "appendix_questions_ja" in raw_data:
                raw_list = raw_data["appendix_questions_ja"]
            else:
                # If the key name is unknown, grab the first value that is a list
                raw_list = next((v for v in raw_data.values() if isinstance(v, list)), [])
        else:
            raw_list = raw_data # It is already a list

        # 3. Standardize keys (Fixes missing 'text_ja' error)
        questions = []
        for item in raw_list:
            # Fix ID: "Q1" -> 1
            raw_id = item.get("id")
            if isinstance(raw_id, str) and raw_id.startswith("Q"):
                q_id = int(raw_id.replace("Q", ""))
            else:
                q_id = raw_id

            # Fix Text: "question" -> "text_ja"
            q_text = item.get("text_ja") or item.get("question")

            questions.append({
                "id": q_id,
                "text_ja": q_text,
                "cis_covered": True
            })

        print(f"✅ Loaded {len(questions)} questions from {QUESTIONS_PATH}")
        print(f"   Sample Q1: {questions[0]['text_ja'][:30]}...")

    except Exception as e:
        print(f"❌ Error parsing JSON: {e}")
        print("   Using minimal questions instead.")
        questions = MINIMAL_QUESTIONS
else:
    print("⚠️ File STILL not found at", QUESTIONS_PATH)
    print("   The code used the 3 dummy questions above.")
    questions = MINIMAL_QUESTIONS

✅ Loaded 62 questions from /content/drive/MyDrive/research/data/questions_62_ja.json
   Sample Q1: 札幌市北区に住んでいます 知人が胆管がんにかかったのですがど...


## 5. CIS scraper (lite or full). Run only when (re)building CIS data.

In [ ]:
import time
import hashlib
import requests
from bs4 import BeautifulSoup

CIS_BASE = "https://ganjoho.jp"
SEEN = set()

def _fname(url):
    h = hashlib.sha256(url.encode()).hexdigest()[:12]
    return CIS_RAW / f"{h}.txt"

def _extract(soup):
    for t in soup(["script", "style"]):
        t.decompose()
    return soup.get_text(separator="\n", strip=True)

def _links(soup):
    for a in soup.find_all("a", href=True):
        h = a["href"].split("#")[0].rstrip("/")
        if h.startswith("/") and not h.startswith("//"):
            yield CIS_BASE + h

def scrape_cis(start=CIS_START, lite=LITE_CRAWL, max_pages=MAX_PAGES_LITE):
    to_visit = [start]
    n = 0
    while to_visit and (not lite or n < max_pages):
        url = to_visit.pop(0)
        if url in SEEN or not url.startswith("https://ganjoho.jp"):
            continue
        SEEN.add(url)
        try:
            r = requests.get(url, timeout=15, headers={"User-Agent": "Mozilla/5.0 (research)"})
            r.raise_for_status()
            soup = BeautifulSoup(r.text, "html.parser")
            text = _extract(soup)
            if len(text) >= 100:
                _fname(url).write_text(text, encoding="utf-8")
                n += 1
            for L in _links(soup):
                if L not in SEEN:
                    to_visit.append(L)
        except Exception as e:
            print(url, e)
        time.sleep(0.4)
    print("CIS pages saved:", n, "under", CIS_RAW)

if RUN_CIS and not list(CIS_RAW.glob("*.txt")):
    scrape_cis(lite=LITE_CRAWL, max_pages=MAX_PAGES_LITE)
elif RUN_CIS:
    print("CIS raw exists. Delete cis/raw/* to re-crawl.")
else:
    print("RUN_CIS=False, skip scraper.")

CIS pages saved: 80 under /content/drive/MyDrive/research/cis/raw


## 6. CIS embed & Chroma index (only if RUN_CIS)

In [ ]:
# --- STEP 6: CIS Embed & Chroma Index (OpenAI Only - Fixed Batching) ---
import chromadb
from chromadb.utils import embedding_functions
import os
import time

# Define Batch Size (Safe limit to avoid 400 Errors)
BATCH_SIZE = 50

if RUN_CIS:
    # 1. Define the OpenAI Embedding Function
    openai_ef = embedding_functions.OpenAIEmbeddingFunction(
        api_key=os.environ.get("OPENAI_API_KEY"),
        model_name=EMBED_MODEL
    )

    # 2. Read and Chunk the text files
    chunks, ids = [], []
    if CIS_RAW.exists():
        for f in sorted(CIS_RAW.glob("*.txt")):
            t = f.read_text(encoding="utf-8")

            for p in t.split("\n\n"):
                p = p.strip()
                if len(p) < 80: continue

                for i in range(0, len(p), 500):
                    c = p[i : i + 500]
                    if len(c) < 80: continue
                    chunks.append(c)
                    ids.append(f"c{len(ids)}")

    # 3. Create/Update the Index (With Batching)
    if chunks:
        print(f"Preparing to embed {len(chunks)} chunks using OpenAI...")

        client = chromadb.PersistentClient(path=str(CIS_INDEX))

        coll = client.get_or_create_collection(
            name="cis",
            embedding_function=openai_ef,
            metadata={"hnsw:space": "cosine"}
        )

        # --- CHANGED: Loop through chunks in small batches ---
        total_batches = (len(chunks) + BATCH_SIZE - 1) // BATCH_SIZE

        for i in range(0, len(chunks), BATCH_SIZE):
            batch_ids = ids[i : i + BATCH_SIZE]
            batch_docs = chunks[i : i + BATCH_SIZE]

            # Send just this batch to OpenAI
            try:
                coll.upsert(ids=batch_ids, documents=batch_docs)
                print(f"  ✅ Batch {(i // BATCH_SIZE) + 1}/{total_batches} indexed ({len(batch_docs)} chunks)")
                time.sleep(0.5) # Polite delay
            except Exception as e:
                print(f"  ❌ Error on batch {(i // BATCH_SIZE) + 1}: {e}")

        print(f"🎉 Finished indexing {len(chunks)} chunks -> {CIS_INDEX}")
    else:
        print("⚠️ No chunks found. Did the Scraper (Step 5) download any pages?")
else:
    print("Skipping Indexing (RUN_CIS=False)")

Preparing to embed 910 chunks using OpenAI...
  ✅ Batch 1/19 indexed (50 chunks)
  ✅ Batch 2/19 indexed (50 chunks)
  ✅ Batch 3/19 indexed (50 chunks)
  ✅ Batch 4/19 indexed (50 chunks)
  ✅ Batch 5/19 indexed (50 chunks)
  ✅ Batch 6/19 indexed (50 chunks)
  ✅ Batch 7/19 indexed (50 chunks)
  ✅ Batch 8/19 indexed (50 chunks)
  ✅ Batch 9/19 indexed (50 chunks)
  ✅ Batch 10/19 indexed (50 chunks)
  ✅ Batch 11/19 indexed (50 chunks)
  ✅ Batch 12/19 indexed (50 chunks)
  ✅ Batch 13/19 indexed (50 chunks)
  ✅ Batch 14/19 indexed (50 chunks)
  ✅ Batch 15/19 indexed (50 chunks)
  ✅ Batch 16/19 indexed (50 chunks)
  ✅ Batch 17/19 indexed (50 chunks)
  ✅ Batch 18/19 indexed (50 chunks)
  ✅ Batch 19/19 indexed (10 chunks)
🎉 Finished indexing 910 chunks -> /content/drive/MyDrive/research/cis/chroma_index


## 7. LLM caller

In [6]:
# --- STEP 7: Define LLM Caller (OpenAI Only) ---
from openai import OpenAI
import os

def call_llm(user_content, model_label):
    """
    model_label: 'gpt4' or 'gpt35'
    """
    # 1. Initialize Client (uses OPENAI_API_KEY from env)
    try:
        client = OpenAI()
    except Exception as e:
        return f"[ERROR: Could not init OpenAI client. {e}]"

    # 2. Select Model
    # These variables (GPT4, GPT35) were defined in Step 2
    if model_label == "gpt4":
        model_name = GPT4
    elif model_label == "gpt35":
        model_name = GPT35
    else:
        model_name = GPT35 # Default fallback

    # 3. Call API
    try:
        response = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": user_content}],
            temperature=0, # Deterministic (Critical for replication)
            max_tokens=1024
        )
        return (response.choices[0].message.content or "").strip()

    except Exception as e:
        print(f"❌ LLM Error ({model_name}): {e}")
        return f"[ERROR: {str(e)}]"

print("✅ LLM function defined. Ready to talk to GPT-4 and GPT-3.5.")

✅ LLM function defined. Ready to talk to GPT-4 and GPT-3.5.


## 8. Retrievers (CIS, Google, Conventional)

In [ ]:
# --- STEP 8: Context Retrievers (with Reranking Upgrade) ---
import chromadb
import os
from openai import OpenAI
from sentence_transformers import CrossEncoder
import numpy as np

# 1. Initialize Reranker (Only once to save time)
if USE_RERANKING:
    print(f"⏳ Loading Reranker model: {RERANK_MODEL_NAME}...")
    # We use a multilingual model because your text is Japanese
    reranker = CrossEncoder(RERANK_MODEL_NAME, max_length=512)
    print("✅ Reranker loaded.")

def cis_context(question_ja: str) -> str:
    """
    Retrieves documents using Vector Search -> Reranking Pipeline.
    """
    if not RUN_CIS:
        return "No relevant information found."

    try:
        # A. Vector Retrieval (Initial Broad Search)
        client = OpenAI()
        response = client.embeddings.create(
            input=[question_ja],
            model=EMBED_MODEL
        )
        query_vec = response.data[0].embedding

        db_client = chromadb.PersistentClient(path=str(CIS_INDEX))
        coll = db_client.get_collection("cis")

        # Get more candidates initially (TOP_N_INITIAL = 20)
        results = coll.query(
            query_embeddings=[query_vec],
            n_results=TOP_N_INITIAL if USE_RERANKING else TOP_K_CIS,
            include=["documents"]
        )

        raw_docs = results["documents"][0] if results["documents"] else []
        if not raw_docs:
            return "No relevant information found."

        # B. Reranking Step (The Upgrade)
        if USE_RERANKING and raw_docs:
            # Create pairs of [Question, Document] for the model to score
            pairs = [[question_ja, doc] for doc in raw_docs]

            # Get relevance scores
            scores = reranker.predict(pairs)

            # Sort documents by score (highest first)
            scored_docs = sorted(zip(scores, raw_docs), key=lambda x: x[0], reverse=True)

            # Keep only the Top K (TOP_K_FINAL = 5)
            final_docs = [doc for score, doc in scored_docs[:TOP_K_FINAL]]
        else:
            final_docs = raw_docs

        return "\n\n".join(final_docs)

    except Exception as e:
        print(f"Error in cis_context: {e}")
        return "No relevant information found (Error)."

# --- UPDATE: Google Retriever with Reranking ---

def google_context(question_ja: str) -> str:
    api_key = os.environ.get("SERPAPI_API_KEY")
    if not api_key: return "No relevant information found."

    try:
        from serpapi import GoogleSearch
        # 1. Broad Search: Get MORE results (Top 20) so the Reranker has options
        params = {
            "q": question_ja,
            "api_key": api_key,
            "num": TOP_N_INITIAL if USE_RERANKING else TOP_K_GOOGLE, # Pull 20 if reranking
            "hl": "ja",
            "gl": "jp"
        }
        search = GoogleSearch(params)
        results = search.get_dict()
        organic = results.get("organic_results", [])

        if not organic: return "No search results found."

        # Format snippets
        raw_snippets = [f"{r.get('title','')}\n{r.get('snippet','')}" for r in organic]

        # 2. Reranking Step
        if USE_RERANKING and raw_snippets:
            # Score them against the question
            pairs = [[question_ja, doc] for doc in raw_snippets]
            scores = reranker.predict(pairs)

            # Sort by highest score
            scored_docs = sorted(zip(scores, raw_snippets), key=lambda x: x[0], reverse=True)

            # Keep only the Best 5
            final_docs = [doc for score, doc in scored_docs[:TOP_K_FINAL]]
        else:
            # Fallback (Original method)
            final_docs = raw_snippets[:TOP_K_GOOGLE]

        return "\n\n".join(final_docs)

    except Exception as e:
        return f"Error: {e}"

def conv_context(question_ja: str) -> str:
    return ""

⏳ Loading Reranker model: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1...


config.json:   0%|          | 0.00/891 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

✅ Reranker loaded.


In [ ]:
import json

# 1. Reload the file you moved
with open(QUESTIONS_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

# 2. Fix the structure (Unwrap the list from the dictionary)
if isinstance(raw_data, dict) and "appendix_questions_ja" in raw_data:
    raw_list = raw_data["appendix_questions_ja"]
else:
    raw_list = raw_data # It might already be a list

# 3. Fix the keys (The notebook needs "text_ja", not "question")
questions = []
for item in raw_list:
    # Fix ID: Convert "Q1" -> 1 (integer)
    raw_id = item.get("id")
    if isinstance(raw_id, str) and raw_id.startswith("Q"):
        q_id = int(raw_id.replace("Q", ""))
    else:
        q_id = raw_id

    # Fix Text: Map "question" -> "text_ja"
    q_text = item.get("text_ja") or item.get("question")

    questions.append({
        "id": q_id,
        "text_ja": q_text,
        "cis_covered": True # Default flag needed for the loop
    })

print(f"✅ SUCCESS: Fixed {len(questions)} questions.")
print(f"Sample format: {questions[0]}")
print("You can now run Step 9!")

✅ SUCCESS: Fixed 62 questions.
Sample format: {'id': 1, 'text_ja': '札幌市北区に住んでいます 知人が胆管がんにかかったのですがどの病院で治療を受けるのがよいですか', 'cis_covered': True}
You can now run Step 9!


## 9. Run all chatbots → `data/results_372.json` (or results_248 if RUN_CIS=False)

In [ ]:
# --- STEP 9: Run All Chatbots (OpenAI Only) ---
import time
import json
import os

# 1. Define the Systems to Test
# The paper tests 3 systems x 2 models = 6 chatbots total
systems = []

# System A: CIS (RAG with Cancer Center Data)
if RUN_CIS and 'cis_context' in globals():
    systems.append(("CIS", "gpt4", cis_context))
    systems.append(("CIS", "gpt35", cis_context))

# System B: Google (RAG with Web Search)
# Only run if SerpAPI key is present
if os.environ.get("SERPAPI_API_KEY") and 'google_context' in globals():
    systems.append(("Google", "gpt4", google_context))
    systems.append(("Google", "gpt35", google_context))
else:
    print("⚠️ Skipping Google Chatbot (No SerpAPI Key found)")

# System C: Conventional (Standard ChatGPT)
if 'conv_context' in globals():
    systems.append(("Conv", "gpt4", conv_context))
    systems.append(("Conv", "gpt35", conv_context))

print(f"\n🚀 Starting Experiment: {len(questions)} Questions x {len(systems)} Chatbots...")

# 2. Main Processing Loop
out = []
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)

for i, q in enumerate(questions):
    # Print progress every 5 questions to keep screen clean
    if i % 5 == 0:
        print(f"Processing Q{q['id']} ({i+1}/{len(questions)})...")

    for sys_name, model_lbl, get_ctx in systems:
        try:
            text = q["text_ja"]

            # 1. Get Context (if RAG)
            if sys_name == "Conv":
                # Conventional has no context
                prompt = text
            else:
                # Retrieve context (CIS or Google)
                ctx = get_ctx(text)
                # Combine into Prompt
                prompt = f"{RAG_INSTRUCTION}\n\nContext:\n{ctx}\n\nQuestion:\n{text}"

            # 2. Call OpenAI
            # We add a small sleep to avoid hitting rate limits too hard
            time.sleep(0.5)
            resp = call_llm(prompt, model_lbl)

            # 3. Store Result
            out.append({
                "question_id": q["id"],
                "system": sys_name,
                "model_label": model_lbl,
                "response": resp
            })

        except Exception as e:
            print(f"  ❌ Error Q{q['id']} [{sys_name}-{model_lbl}]: {e}")

    # 3. Save constantly (prevents data loss)
    with open(RESULTS_PATH, "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=2)

print(f"\n✅ DONE! All results saved to: {RESULTS_PATH}")


🚀 Starting Experiment: 62 Questions x 6 Chatbots...
Processing Q1 (1/62)...
Processing Q6 (6/62)...
Processing Q11 (11/62)...
Processing Q16 (16/62)...
Processing Q21 (21/62)...
Processing Q26 (26/62)...
Processing Q31 (31/62)...
Processing Q36 (36/62)...
Processing Q41 (41/62)...
Processing Q46 (46/62)...
Processing Q51 (51/62)...
Processing Q56 (56/62)...
Processing Q61 (61/62)...

✅ DONE! All results saved to: /content/drive/MyDrive/research/data/results_372.json


In [ ]:
# --- STEP 9: RESUME All Chatbots (Don't start over!) ---
import time
import json
import os

# 1. Define Systems
systems = []
if RUN_CIS and 'cis_context' in globals():
    systems.append(("CIS", "gpt4", cis_context))
    systems.append(("CIS", "gpt35", cis_context))
if os.environ.get("SERPAPI_API_KEY") and 'google_context' in globals():
    systems.append(("Google", "gpt4", google_context))
    systems.append(("Google", "gpt35", google_context))
if 'conv_context' in globals():
    systems.append(("Conv", "gpt4", conv_context))
    systems.append(("Conv", "gpt35", conv_context))

# 2. Load Existing Progress
out = []
completed_ids = set()

if RESULTS_PATH.exists():
    try:
        with open(RESULTS_PATH, "r", encoding="utf-8") as f:
            out = json.load(f)

        # Count how many system-responses we have for each question
        # We need 6 responses (3 systems x 2 models) to consider a question "done"
        q_counts = {}
        for row in out:
            qid = row['question_id']
            q_counts[qid] = q_counts.get(qid, 0) + 1

        # If a question has all 6 answers, mark it as done
        required_responses = len(systems)
        for qid, count in q_counts.items():
            if count >= required_responses:
                completed_ids.add(qid)

        print(f"🔄 Resuming... Found {len(out)} saved responses.")
        print(f"✅ Skipping {len(completed_ids)} fully completed questions.")

    except Exception as e:
        print(f"⚠️ Could not read existing file: {e}. Starting fresh.")
else:
    print("🆕 No existing results found. Starting fresh.")

# 3. Main Loop (With Skip Logic)
print(f"\n🚀 Processing remaining questions...")

RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)

for i, q in enumerate(questions):
    # SKIP logic
    if q['id'] in completed_ids:
        continue

    print(f"Processing Q{q['id']} ({i+1}/{len(questions)})...")

    for sys_name, model_lbl, get_ctx in systems:
        # Check if we already have this SPECIFIC answer (partial recovery)
        already_done = False
        for existing in out:
            if (existing['question_id'] == q['id'] and
                existing['system'] == sys_name and
                existing['model_label'] == model_lbl):
                already_done = True
                break

        if already_done:
            continue

        try:
            text = q["text_ja"]

            if sys_name == "Conv":
                prompt = text
            else:
                ctx = get_ctx(text)
                prompt = f"{RAG_INSTRUCTION}\n\nContext:\n{ctx}\n\nQuestion:\n{text}"

            time.sleep(0.5)
            resp = call_llm(prompt, model_lbl)

            out.append({
                "question_id": q["id"],
                "system": sys_name,
                "model_label": model_lbl,
                "response": resp
            })

        except Exception as e:
            print(f"  ❌ Error Q{q['id']} [{sys_name}-{model_lbl}]: {e}")

    # Save constantly
    with open(RESULTS_PATH, "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=2)

print(f"\n✅ DONE! Results saved to: {RESULTS_PATH}")

🔄 Resuming... Found 372 saved responses.
✅ Skipping 62 fully completed questions.

🚀 Processing remaining questions...

✅ DONE! Results saved to: /content/drive/MyDrive/p9_gpt/data/results_372.json


In [ ]:
# --- STEP 10: Automated Evaluation (LLM-as-a-Judge) ---
import pandas as pd
import json
import time
from openai import OpenAI

# 1. Load Results
if not RESULTS_PATH.exists():
    print("❌ No results file found. Run Step 9 first.")
else:
    with open(RESULTS_PATH, "r", encoding="utf-8") as f:
        results_data = json.load(f)
    print(f"Loaded {len(results_data)} responses for evaluation.")

    # 2. Define the Judge Function
    def evaluate_faithfulness(question, context, response):
        """
        Uses GPT-4o (or GPT-4) to judge if the response contains hallucinations.
        """
        if "I don't know" in response or "提供された情報" in response:
            return 0, "Refusal" # Not a hallucination, just a refusal

        judge_prompt = f"""
        You are an impartial judge evaluating a chatbot's response to a medical question.

        CONTEXT:
        {context}

        QUESTION:
        {question}

        RESPONSE:
        {response}

        TASK:
        Does the RESPONSE contain any medical claims that are NOT supported by the CONTEXT?
        - Ignore stylistic differences.
        - If the response uses outside knowledge not in context, it is a Hallucination (1).
        - If the response is fully supported by context, it is Faithful (0).

        OUTPUT FORMAT:
        Return ONLY a JSON object: {{"hallucination": 0 or 1, "reason": "short explanation"}}
        """

        try:
            client = OpenAI()
            eval_resp = client.chat.completions.create(
                model="gpt-4o", # Use your best available model here
                messages=[{"role": "user", "content": judge_prompt}],
                temperature=0,
                response_format={"type": "json_object"}
            )
            result = json.loads(eval_resp.choices[0].message.content)
            return result["hallucination"], result["reason"]
        except Exception as e:
            return 0, f"Error: {e}"

    # 3. Run Evaluation Loop
    print("👨‍⚖️ Starting LLM Grading (This may take time)...")
    graded_results = []

    # We re-fetch context for grading purposes
    # Note: In a real run, you might want to save the retrieved context in Step 9 to avoid re-retrieving here.
    # For now, we will just grade the text.

    for i, row in enumerate(results_data):
        if i % 10 == 0: print(f"Grading {i}/{len(results_data)}...")

        # We need the context to judge faithfulness.
        # Since we didn't save context in Step 9, we will re-retrieve it quickly here.
        if row['system'] == 'CIS':
            ctx = cis_context(questions[row['question_id']-1]['text_ja'])
        elif row['system'] == 'Google':
            ctx = google_context(questions[row['question_id']-1]['text_ja'])
        else:
            ctx = "" # Conventional has no context -> Checking for hallucinations against general knowledge is harder without ground truth.
                     # Usually we skip Hallucination-check for Conventional or assume everything is 0.

        if row['system'] != 'Conv':
            score, reason = evaluate_faithfulness(
                questions[row['question_id']-1]['text_ja'],
                ctx,
                row['response']
            )
        else:
            score, reason = 0, "Skipped (Conventional)"

        new_row = row.copy()
        new_row['is_hallucination'] = score
        new_row['judge_reason'] = reason
        graded_results.append(new_row)

    # 4. Save & Visualize
    df_graded = pd.DataFrame(graded_results)
    save_path = DATA_DIR / "results_graded.csv"
    df_graded.to_csv(save_path, index=False)
    print(f"✅ Grading Complete. Saved to {save_path}")

    # Show Summary Stats
    print("\n--- 📊 Evaluation Report ---")
    summary = df_graded.groupby(['system', 'model_label'])['is_hallucination'].mean() * 100
    print("Hallucination Rates (%):")
    print(summary)

Loaded 372 responses for evaluation.
👨‍⚖️ Starting LLM Grading (This may take time)...
Grading 0/372...
Grading 10/372...
Grading 20/372...
Grading 30/372...
Grading 40/372...
Grading 50/372...
Grading 60/372...
Grading 70/372...
Grading 80/372...
Grading 90/372...
Grading 100/372...
Grading 110/372...
Grading 120/372...
Grading 130/372...
Grading 140/372...
Grading 150/372...
Grading 160/372...
Grading 170/372...
Grading 180/372...
Grading 190/372...
Grading 200/372...
Grading 210/372...
Grading 220/372...
Grading 230/372...
Grading 240/372...
Grading 250/372...
Grading 260/372...
Grading 270/372...
Grading 280/372...
Grading 290/372...
Grading 300/372...
Grading 310/372...
Grading 320/372...
Grading 330/372...
Grading 340/372...
Grading 350/372...
Grading 360/372...
Grading 370/372...
✅ Grading Complete. Saved to /content/drive/MyDrive/research/data/results_graded.csv

--- 📊 Evaluation Report ---
Hallucination Rates (%):
system  model_label
CIS     gpt35          19.354839
        gp

## 10. Download results (or copy from Drive)

In [ ]:
if IN_COLAB:
    from google.colab import files
    files.download(str(RESULTS_PATH))
else:
    print("Local run; results at", RESULTS_PATH)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
# --- STEP 11: Define Self-Correcting RAG (The Improvement) ---
import json

def self_correcting_rag(question_ja, get_context_fn, model_label="gpt4"):
    """
    1. Retrieves context.
    2. Generates a draft answer.
    3. Self-Corrects: Asks the LLM to verify if the draft is supported by context.
    4. If unsupported, regenerates a strict version.
    """
    # 1. Retrieve Context
    context = get_context_fn(question_ja)

    # 2. Generate Draft
    draft_prompt = f"""
    Use the following context to answer the question.
    If you don't know, say you don't know.

    Context:
    {context}

    Question:
    {question_ja}
    """
    draft_response = call_llm(draft_prompt, model_label)

    # 3. Self-Correction / Verification Step
    # We ask the model to act as a strict auditor of its own work.
    verify_prompt = f"""
    Context:
    {context}

    Draft Answer:
    {draft_response}

    Task:
    Check if the Draft Answer contains any factual claims NOT supported by the Context.
    - If the answer is fully supported, respond with just "PASSED".
    - If there are hallucinations, list them briefly.
    """
    verification = call_llm(verify_prompt, model_label)

    # 4. Logic: Keep or Refine?
    if "PASSED" in verification:
        return draft_response # Trust the draft
    else:
        # 5. Refinement Step (The Fix)
        refine_prompt = f"""
        The previous draft answer contained information not in the context.

        Context:
        {context}

        Critique of Draft:
        {verification}

        Task:
        Rewrite the answer to be 100% faithful to the context.
        Remove any unsupported information mentioned in the critique.
        """
        final_response = call_llm(refine_prompt, model_label)
        return final_response

print("✅ Self-Correcting RAG system defined.")

✅ Self-Correcting RAG system defined.


In [2]:
# --- STEP 12: Run Experiment with Self-Correcting RAG ---
# We will test the "Self-Correcting" system against the standard CIS context
improved_systems = [
    ("CIS_Corrected", "gpt4", cis_context)
    # You can add Google_Corrected here if you have the API key active
]

improved_out = []
print(f"🚀 Testing Self-Correcting System on {len(questions)} questions...")

for i, q in enumerate(questions):
    if i % 10 == 0: print(f"Processing Q{q['id']}...")

    for sys_name, model_lbl, get_ctx in improved_systems:
        try:
            # Call the new function defined in Step 11
            resp = self_correcting_rag(q['text_ja'], get_ctx, model_lbl)

            improved_out.append({
                "question_id": q['id'],
                "system": sys_name,
                "model_label": model_lbl,
                "response": resp
            })
        except Exception as e:
            print(f"  ❌ Error Q{q['id']}: {e}")

# Save results
IMPROVED_RESULTS_PATH = DATA_DIR / "results_improved.json"
with open(IMPROVED_RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(improved_out, f, ensure_ascii=False, indent=2)

print(f"✅ Improved results saved to {IMPROVED_RESULTS_PATH}")

NameError: name 'cis_context' is not defined

In [7]:
# ==========================================
# MASTER CELL: Self-Correcting RAG + Evaluation
# ==========================================

import json
import os
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from openai import OpenAI

# --- PART 1: SAFTEY CHECKS (Do we have what we need?) ---
# 1. Check for Questions
if 'questions' not in globals():
    try:
        with open(DATA_DIR / "questions_62_ja.json", "r", encoding="utf-8") as f:
            raw = json.load(f)
            # Handle potential nesting
            if isinstance(raw, dict):
                raw_list = raw.get("appendix_questions_ja", list(raw.values())[0])
            else:
                raw_list = raw
            questions = []
            for item in raw_list:
                qid = item.get("id")
                if isinstance(qid, str): qid = int(qid.replace("Q", ""))
                questions.append({"id": qid, "text_ja": item.get("text_ja") or item.get("question")})
        print(f"✅ Loaded {len(questions)} questions from file.")
    except Exception as e:
        print(f"⚠️ Could not load questions: {e}. creating dummy variable.")
        questions = []

# 2. Check for Context Function (cis_context)
# If Step 8 was skipped, we redefine it here.
if 'cis_context' not in globals():
    print("🔄 redefining cis_context function...")
    import chromadb
    # Ensure constants exist
    if 'CIS_INDEX' not in globals(): CIS_INDEX = PROJECT_ROOT / "cis" / "chroma_index"
    if 'EMBED_MODEL' not in globals(): EMBED_MODEL = "text-embedding-ada-002"

    def cis_context(question_ja: str) -> str:
        try:
            client = OpenAI()
            resp = client.embeddings.create(input=[question_ja], model=EMBED_MODEL)
            query_vec = resp.data[0].embedding

            db = chromadb.PersistentClient(path=str(CIS_INDEX))
            coll = db.get_collection("cis")
            res = coll.query(query_embeddings=[query_vec], n_results=5, include=["documents"])
            docs = res["documents"][0] if res["documents"] else []
            return "\n\n".join(docs) if docs else "No info."
        except Exception as e:
            return f"Error retrieving context: {e}"

# 3. Check for Judge Function (evaluate_faithfulness)
if 'evaluate_faithfulness' not in globals():
    def evaluate_faithfulness(question, context, response):
        if "I don't know" in response or "情報がありません" in response:
            return 0, "Abstention"

        prompt = f"""
        CONTEXT:\n{context}\n\nRESPONSE:\n{response}\n
        TASK: Does the RESPONSE contain medical claims NOT supported by the CONTEXT?
        Output JSON: {{"hallucination": 0 or 1}}
        """
        try:
            client = OpenAI()
            res = client.chat.completions.create(
                model="gpt-4o", # Use a smart model for judging
                messages=[{"role": "user", "content": prompt}],
                temperature=0, response_format={"type": "json_object"}
            )
            return json.loads(res.choices[0].message.content)["hallucination"], "Graded"
        except:
            return 0, "Error"

# --- PART 2: THE SELF-CORRECTING LOGIC (Abstention Focused) ---

def self_correcting_rag(question_ja, get_context_fn, model_label="gpt4"):
    """
    1. Drafts an answer.
    2. Self-Verifies: If unsupported, it rewrites or ABSTAINS.
    """
    context = get_context_fn(question_ja)

    # 1. Draft
    draft_resp = call_llm(f"Context:\n{context}\n\nQuestion:\n{question_ja}", model_label)

    # 2. Verify (The "Judge" step)
    verify_prompt = f"""
    Context:\n{context}\n\nDraft Answer:\n{draft_resp}\n
    Task: Check for hallucinations.
    If the answer is fully supported, say "PASSED".
    If it contains unsupported info, list the specific errors.
    """
    critique = call_llm(verify_prompt, model_label)

    if "PASSED" in critique:
        return draft_resp
    else:
        # 3. Correction / Abstention
        # We explicitly tell it to ABSTAIN if it can't fix it.
        fix_prompt = f"""
        Context:\n{context}\n\nCritique:\n{critique}\n
        Task: Rewrite the answer. Remove all hallucinations.
        CRITICAL: If the Context is not sufficient to answer the question,
        you MUST explicitly say "I don't know" or "The provided text does not contain this information."
        Do not guess.
        """
        return call_llm(fix_prompt, model_label)

# --- PART 3: RUNNING THE EXPERIMENT ---
print(f"🚀 Starting Self-Correcting Experiment on {len(questions)} questions...")
improved_results = []
save_path = DATA_DIR / "results_improved.json"

# Check if we can resume specific progress (Optional logic)
processed_ids = set()
if save_path.exists():
    with open(save_path, "r", encoding="utf-8") as f:
        existing = json.load(f)
        improved_results = existing
        processed_ids = {row['question_id'] for row in existing}
    print(f"🔄 Resuming... Skipping {len(processed_ids)} already corrected questions.")

count = 0
for q in questions:
    if q['id'] in processed_ids: continue # Skip done

    try:
        # Run the new system
        resp = self_correcting_rag(q['text_ja'], cis_context, "gpt4")

        improved_results.append({
            "question_id": q['id'],
            "system": "CIS_Corrected",
            "model_label": "gpt4",
            "response": resp
        })
        count += 1

        # Save every 5 questions to be safe
        if count % 5 == 0:
            with open(save_path, "w", encoding="utf-8") as f:
                json.dump(improved_results, f, ensure_ascii=False, indent=2)
            print(f"  Saved progress at Q{q['id']}...")

    except Exception as e:
        print(f"❌ Error on Q{q['id']}: {e}")

# Final Save
with open(save_path, "w", encoding="utf-8") as f:
    json.dump(improved_results, f, ensure_ascii=False, indent=2)
print("✅ Experiment Done.")

# --- PART 4: EVALUATE & PLOT ---
print("👨‍⚖️ Grading new answers...")
graded_data = []
for row in improved_results:
    q_text = next((item["text_ja"] for item in questions if item["id"] == row["question_id"]), "")
    ctx = cis_context(q_text)
    score, _ = evaluate_faithfulness(q_text, ctx, row["response"])
    row["is_hallucination"] = score
    graded_data.append(row)

# Load OLD results (to compare without spending tokens)
try:
    df_old = pd.read_csv(DATA_DIR / "results_graded.csv")
    # Filter for just the basic CIS GPT4 to compare against
    df_old = df_old[df_old['system'] == 'CIS']
except:
    df_old = pd.DataFrame()

# Combine
df_new = pd.DataFrame(graded_data)
df_final = pd.concat([df_old, df_new], ignore_index=True)

# Plot
if not df_final.empty:
    plt.figure(figsize=(10, 6))
    sns.barplot(data=df_final, x='system', y='is_hallucination', hue='model_label', palette='viridis', errorbar=None)
    plt.title("Hallucination Rate: Standard CIS vs. Self-Correcting")
    plt.ylabel("Hallucination Rate (0-1)")
    plt.show()

    print("Rates (%):")
    print(df_final.groupby(['system'])['is_hallucination'].mean() * 100)
else:
    print("⚠️ No data to plot.")

✅ Loaded 62 questions from file.
🔄 redefining cis_context function...
🚀 Starting Self-Correcting Experiment on 62 questions...
  Saved progress at Q5...
❌ LLM Error (gpt-4): Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4 in organization org-f4GTYzgpdCgWozBTrBdRqwNZ on tokens per min (TPM): Limit 10000, Used 8262, Requested 1780. Please try again in 251ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}
  Saved progress at Q10...
  Saved progress at Q15...
❌ LLM Error (gpt-4): Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}
❌ LLM Error (gpt-4): Error code: 429 - {'error': {'message': 'You exceeded your current quota, pl

KeyboardInterrupt: 